In [3]:
import os
import pandas as pd
import gspread
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta

def connect_to_sheet():
    SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
    creds = None
    if os.path.exists(r'C:\Users\mochamad.ilmawan\token.json'):
        creds = Credentials.from_authorized_user_file(r'C:\Users\mochamad.ilmawan\token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                r'D:\OneDrive\Documents\Direct Selling\MONITORING\Monitoring 2023\client_secret_738271294618-6g8e0hle4jpq8nau1d7b3q9jfsgp0ruk.apps.googleusercontent.com.json', 
                SCOPES
            )
            creds = flow.run_local_server(port=0)
        with open(r'C:\Users\mochamad.ilmawan\token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

def update_sheet(spreadsheetId, spread_range, scopes, creds, df):
    df_clean = df.copy()
    for col in df_clean.columns:
        if pd.api.types.is_datetime64_any_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].dt.strftime('%Y-%m-%d')
            
    df_clean = df_clean.astype(object).fillna('')
    header = df_clean.columns.tolist()
    data = df_clean.values.tolist()
    
    clean_data = []
    for row in data:
        clean_row = []
        for cell in row:
            if isinstance(cell, (datetime, pd.Timestamp)):
                clean_row.append(cell.strftime('%Y-%m-%d'))
            elif pd.isna(cell):
                clean_row.append('')
            else:
                clean_row.append(cell)
        clean_data.append(clean_row)
        
    values_with_header = [header] + clean_data
    try:
        service = build('sheets', 'v4', credentials=creds)
        service.spreadsheets().values().clear(spreadsheetId=spreadsheetId, range=spread_range, body={}).execute()
        body = {'values': values_with_header}
        result = service.spreadsheets().values().update(
            spreadsheetId=spreadsheetId, range=spread_range, valueInputOption="USER_ENTERED", body=body
        ).execute()
        print(f"{result.get('updatedCells')} cells updated (sheet replaced).")
        return result
    except HttpError as error:
        print(f"An error occurred: {error}")
        return error

def get_excel_data(file_path, sheet_name, usecols, skiprows, nrows, header_row):
    try:
        df = pd.read_excel(
            file_path,
            sheet_name=sheet_name,
            usecols=usecols,
            skiprows=skiprows - 1, 
            nrows=nrows,
            header=header_row
        )
        return df
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return None

def extract_and_normalize_data(file_path):
    combined_df = get_excel_data(file_path, 'update 19 jan 26 (UPtes (2)', "A:M, BH:BI, BQ:CV", 2, 1000, 1)
    
    if combined_df is not None:
        # --- STEP 1: Rebuild headers using Row 0 ('Plan'/'Real') and Timestamps ---
        new_columns = []
        current_date = None
        
        indicator_row = combined_df.iloc[0].fillna('').astype(str).str.strip().str.lower()
        
        for col_name, indicator in zip(combined_df.columns, indicator_row):
            col_str = str(col_name).strip()
            
            if '2026' in col_str and 'unnamed' not in col_str.lower():
                current_date = col_str.split()[0]
            
            if current_date and ( 'plan' in indicator or 'real' in indicator ):
                clean_type = 'Planning' if 'plan' in indicator else 'Realisasi'
                new_columns.append(f"{current_date}_{clean_type}")
            else:
                # Standardize strings to clean up metadata checks
                new_columns.append(col_str.lower().replace('.', '').strip())
                
        combined_df.columns = new_columns
        
        # Drop row 0 now that date titles are securely merged
        combined_df = combined_df.drop(combined_df.index[0]).reset_index(drop=True)
        
        # --- STEP 2: Handle explicit mapping normalization strings ---
        rename_dict = {
            'no': 'No',
            '3 no project / wbs': 'Project Code',
            'project code / wbs': 'Project Code',
            'plant': 'Plant',
            'semen / non semen': 'Semen Non Semen',
            'semen  non semen': 'Semen Non Semen',
            '2 item capex': 'Nama Capex',
            'nama capex': 'Nama Capex',
            'tahun capex': 'Tahun Capex',
            '4 entitas': 'Entitas',
            'entitas': 'Entitas',
            'tipe capex (ko/pko/m/po/str)': 'Tipe Capex',
            'coc/new/multiyears': 'COC New Multiyears',
            'coc / new': 'COC New',
            'tahapan': 'Tahapan',
            'status': 'Status',
            'total nilai investasi': 'Total Nilai Investasi',
            'rencana spending 2025': 'Rencana Spending 2025',
            'rencana spending 2026': 'Rencana Spending 2026'
        }
        combined_df = combined_df.rename(columns=rename_dict)
        
        for col in ['Nama Capex', 'Project Code']:
            if col in combined_df.columns:
                combined_df[col] = combined_df[col].fillna('belum diinput').astype(str).str.strip()
        # --- STEP 3: Normalize using 'Nama Capex' as the primary tracking ID key ---
        if 'Nama Capex' not in combined_df.columns:
            print("Error: 'Nama Capex' column could not be found or resolved.")
            return combined_df

        # Isolate wide monthly date target variables
        value_vars = [col for col in combined_df.columns if '_planning' in col.lower() or '_realisasi' in col.lower()]
        
        if not value_vars:
            print("Warning: No structured timeline targets matched to normalize.")
            return combined_df

        # Melt wide headers using 'Nama Capex' to prevent structure breaks or dropped rows
        df_melted = pd.melt(
            combined_df,
            id_vars=['Nama Capex'],
            value_vars=value_vars,
            var_name='Metric_Raw',
            value_name='Value'
        )
        
        df_melted[['Date_Raw', 'Type']] = df_melted['Metric_Raw'].str.split('_', expand=True)
        
        # --- STEP 4: Convert date strings to dynamic end-of-month (dd/mm/yyyy) ---
        def get_end_of_month(date_str):
            try:
                dt = pd.to_datetime(date_str)
                end_date = dt + pd.offsets.MonthEnd(0)
                return end_date.strftime('%d/%m/%Y')
            except:
                return 'Unknown'
                
        df_melted['Month'] = df_melted['Date_Raw'].apply(get_end_of_month)
        df_melted = df_melted[df_melted['Month'] != 'Unknown']
        
        # Pivot the timeline variables back out cleanly
        df_pivot = df_melted.pivot_table(
            index=['Nama Capex', 'Month'],
            columns='Type',
            values='Value',
            aggfunc='first'
        ).reset_index()
        
        # --- STEP 5: Re-merge project attributes safely back into the final table ---
        metadata_cols = [
            'No', 'Project Code', 'Plant', 'Semen Non Semen', 'Entitas', 
            'Nama Capex', 'Tahun Capex', 'Tipe Capex', 'COC New Multiyears', 
            'COC New', 'Tahapan', 'Status', 'Total Nilai Investasi', 
            'Rencana Spending 2025', 'Rencana Spending 2026'
        ]
        existing_meta = [col for col in metadata_cols if col in combined_df.columns]
        
        # Drop duplicates on our key from the raw data to ensure accurate mapping alignment
        df_meta_unique = combined_df[existing_meta].drop_duplicates(subset=['Nama Capex'])
        
        # Left merge everything onto our pivoted layout frame
        df_clean = pd.merge(df_pivot, df_meta_unique, on='Nama Capex', how='left')
        
        # Force fill completely empty variables requested by schema to avoid spreadsheet generation drops
        for col in metadata_cols + ['Planning', 'Realisasi']:
            if col not in df_clean.columns:
                df_clean[col] = ''
                
        if 'Group' not in df_clean.columns:
            df_clean['Group'] = 'Other'
            
        df_clean.columns.name = None
        
        # --- STEP 6: Enforce absolute structural sorting order layout requested ---
        target_order = [
            'No', 'Project Code', 'Plant', 'Semen Non Semen', 'Entitas', 
            'Nama Capex', 'Tahun Capex', 'Tipe Capex', 'COC New Multiyears', 
            'COC New', 'Tahapan', 'Status', 'Total Nilai Investasi', 
            'Rencana Spending 2025', 'Rencana Spending 2026', 'Month', 
            'Planning', 'Realisasi', 'Group'
        ]
        
        df_final = df_clean.reindex(columns=target_order)
        print("\nSuccessfully restructured data framework using 'Nama Capex' as the ID link!")
        return df_final
    else:
        print("\nNo data collected. Please verify your file paths.")
        return None

def main():
    clean_df = extract_and_normalize_data(r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Capex\PROFILE CAPEX RKAP 1565 (5.  Mar 2026)c-2-4 (CHECK).xlsx")
    
    if clean_df is not None and not clean_df.empty:
        display(clean_df.head(10))
        print(f"Final Normalized Data Shape: {clean_df.shape[0]} rows x {clean_df.shape[1]} columns")
        
        creds = connect_to_sheet()
        update_sheet('1ckgGhsbbHGYfpqXtak_EG79ElY00bTvAADbNySRMa1s', 'Sheet1', None, creds, clean_df)
    else:
        print("Data processing returned an empty structure. Step skipped.")

if __name__ == '__main__':
    main()


Successfully restructured data framework using 'Nama Capex' as the ID link!


,No,Project Code,Plant,Semen Non Semen,Entitas,Nama Capex,Tahun Capex,Tipe Capex,COC New Multiyears,COC New,Tahapan,Status,Total Nilai Investasi,Rencana Spending 2025,Rencana Spending 2026,Month,Planning,Realisasi,Group
0,250.0,belum diinput,NaN,Semen,Semen Padang,"""Revitalisasi Air Lock Equipment Jalur Raw Mil...",2026.0,PO,NEW,NEW,FEL 2,Evaluation by KI,1.700000e+10,NaN,1700000000,31/12/2026,1700000000,NaN,Other
1,3.0,P2-25327,NaN,SEMEN,Holding,-C Replace successfactors ICT sf,2025.0,PKO,COC,COC,NaN,Project,6.500000e+09,645000000,2100000000,30/04/2026,2100000000,752500000,Other
2,1066.0,SN2.2304,NaN,Semen,SBI,AFR Drilling waste management service,2024.0,PO,COC,COC,NaN,Project,NaN,NaN,2700000000,28/02/2026,NaN,742452603,Other
3,1066.0,SN2.2304,NaN,Semen,SBI,AFR Drilling waste management service,2024.0,PO,COC,COC,NaN,Project,NaN,NaN,2700000000,30/04/2026,NaN,0,Other
4,1066.0,SN2.2304,NaN,Semen,SBI,AFR Drilling waste management service,2024.0,PO,COC,COC,NaN,Project,NaN,NaN,2700000000,31/03/2026,NaN,300002548,Other
5,1066.0,SN2.2304,NaN,Semen,SBI,AFR Drilling waste management service,2024.0,PO,COC,COC,NaN,Project,NaN,NaN,2700000000,31/05/2026,NaN,424293000,Other
6,1066.0,SN2.2304,NaN,Semen,SBI,AFR Drilling waste management service,2024.0,PO,COC,COC,NaN,Project,NaN,NaN,2700000000,31/12/2026,2700000000,NaN,Other
7,735.0,SB-25010,NaN,Non Semen,IKSG,"AIR SHAFT ( laminasi, starkon, remax, janson )",2025.0,PKO,COC,COC,NaN,Project,4.000000e+08,NaN,400000000,30/09/2026,400000000,NaN,Other
8,725.0,belum diinput,NaN,Semen,GhoPo,Additional Preduster PKC 470RM1 Untuk penuruna...,2026.0,PO,NEW,NEW,FEL 1,Revisi FS by User,4.000000e+09,NaN,3000000000,31/12/2026,3000000000,NaN,Other
9,734.0,SB-25015,NaN,Non Semen,IKSG,"BAG CHECK SYSTEM ( BCS ) Line 3,4",2025.0,PKO,COC,COC,NaN,Project,5.000000e+08,NaN,500000000,31/08/2026,500000000,NaN,Other


Final Normalized Data Shape: 4687 rows x 19 columns


C:\Users\mochamad.ilmawan\AppData\Local\Temp\ipykernel_32704\1985748374.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean = df_clean.astype(object).fillna('')


89072 cells updated (sheet replaced).
